# BBO Challenge – GP + UCB Strategy (Week 2)

This notebook reproduces a simple Gaussian Process (GP) + Upper Confidence Bound (UCB) strategy for proposing Week 2 query points for Functions F1–F8.


In [7]:
import numpy as np
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.exceptions import ConvergenceWarning
import warnings

data_dir = Path('data/initial_data')

def load_function_data(tag):
    X = np.load(data_dir / f"{tag}_initial_inputs.npy")
    y = np.load(data_dir / f"{tag}_initial_outputs.npy")
    print(f"{tag}: X shape = {X.shape}, y shape = {y.shape}")
    return X, y


## Last week’s query points and outputs

In [8]:
last_X = {
    'F1': np.array([0.759779, 0.804108]),
    'F2': np.array([0.717915, 0.002064]),
    'F3': np.array([0.994942, 0.051967, 0.792526]),
    'F4': np.array([0.404477, 0.413254, 0.303108, 0.434359]),
    'F5': np.array([0.940282, 0.064909, 0.998637, 0.996528]),
    'F6': np.array([0.379776, 0.684419, 0.068171, 0.984146, 0.290503]),
    'F7': np.array([0.015298, 0.590564, 0.658382, 0.751493, 0.425786, 0.776978]),
    'F8': np.array([0.034492, 0.437681, 0.011459, 0.323393, 0.989664, 0.045780, 0.097175, 0.702465]),
}

last_y = {
    'F1': 3.1007222612420183e-34,
    'F2': 0.5697925224470959,
    'F3': -0.15184421067439557,
    'F4': -0.7158150747637886,
    'F5': 3653.9508244023123,
    'F6': -1.362061899095797,
    'F7': 0.23953469687403112,
    'F8': 9.5899381138985,
}


## Build updated datasets (initial + last week’s point)

In [9]:
F1_X, F1_y = load_function_data('F1')
F2_X, F2_y = load_function_data('F2')
F3_X, F3_y = load_function_data('F3')
F4_X, F4_y = load_function_data('F4')
F5_X, F5_y = load_function_data('F5')
F6_X, F6_y = load_function_data('F6')
F7_X, F7_y = load_function_data('F7')
F8_X, F8_y = load_function_data('F8')

F1_X2, F1_y2 = np.vstack([F1_X, last_X['F1']]), np.append(F1_y, last_y['F1'])
F2_X2, F2_y2 = np.vstack([F2_X, last_X['F2']]), np.append(F2_y, last_y['F2'])
F3_X2, F3_y2 = np.vstack([F3_X, last_X['F3']]), np.append(F3_y, last_y['F3'])
F4_X2, F4_y2 = np.vstack([F4_X, last_X['F4']]), np.append(F4_y, last_y['F4'])
F5_X2, F5_y2 = np.vstack([F5_X, last_X['F5']]), np.append(F5_y, last_y['F5'])
F6_X2, F6_y2 = np.vstack([F6_X, last_X['F6']]), np.append(F6_y, last_y['F6'])
F7_X2, F7_y2 = np.vstack([F7_X, last_X['F7']]), np.append(F7_y, last_y['F7'])
F8_X2, F8_y2 = np.vstack([F8_X, last_X['F8']]), np.append(F8_y, last_y['F8'])

print('Updated shapes:')
for tag, X2, y2 in [
    ('F1', F1_X2, F1_y2),
    ('F2', F2_X2, F2_y2),
    ('F3', F3_X2, F3_y2),
    ('F4', F4_X2, F4_y2),
    ('F5', F5_X2, F5_y2),
    ('F6', F6_X2, F6_y2),
    ('F7', F7_X2, F7_y2),
    ('F8', F8_X2, F8_y2),
]:
    print(f"{tag}: {X2.shape}, {y2.shape}")


F1: X shape = (10, 2), y shape = (10,)
F2: X shape = (10, 2), y shape = (10,)
F3: X shape = (15, 3), y shape = (15,)
F4: X shape = (30, 4), y shape = (30,)
F5: X shape = (20, 4), y shape = (20,)
F6: X shape = (20, 5), y shape = (20,)
F7: X shape = (30, 6), y shape = (30,)
F8: X shape = (40, 8), y shape = (40,)
Updated shapes:
F1: (11, 2), (11,)
F2: (11, 2), (11,)
F3: (16, 3), (16,)
F4: (31, 4), (31,)
F5: (21, 4), (21,)
F6: (21, 5), (21,)
F7: (31, 6), (31,)
F8: (41, 8), (41,)


## GP + UCB helper functions

In [10]:
def suggest_point_ucb(X, y, n_candidates, beta=2.0, random_state=0):
    """Fit a GP surrogate and propose a new point via UCB.

    UCB(x) = mean(x) + beta * std(x)
    """
    rng = np.random.RandomState(random_state)
    n_dim = X.shape[1]

    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(n_dim), length_scale_bounds=(0.05, 2.0))
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-2))
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=random_state,
        n_restarts_optimizer=1,
    )

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)

    cand = rng.uniform(0.0, 1.0, size=(n_candidates, n_dim))
    mean, std = gp.predict(cand, return_std=True)

    ucb = mean + beta * std
    best_idx = np.argmax(ucb)
    return cand[best_idx], float(mean[best_idx]), float(std[best_idx])

def format_point(x):
    return '-'.join(f"{float(v):.6f}" for v in x)


## Propose Week 2 query points with GP + UCB

In [11]:
configs = {
    'F1': (F1_X2, F1_y2, 30000),
    'F2': (F2_X2, F2_y2, 30000),
    'F3': (F3_X2, F3_y2, 25000),
    'F4': (F4_X2, F4_y2, 25000),
    'F5': (F5_X2, F5_y2, 25000),
    'F6': (F6_X2, F6_y2, 20000),
    'F7': (F7_X2, F7_y2, 15000),
    'F8': (F8_X2, F8_y2, 12000),
}

suggestions = {}

for tag, (X2, y2, n_cand) in configs.items():
    x_star, m_star, s_star = suggest_point_ucb(X2, y2, n_candidates=n_cand, beta=2.0, random_state=123)
    suggestions[tag] = {
        'x': x_star,
        'formatted': format_point(x_star),
        'mean_pred': m_star,
        'std_pred': s_star,
    }
    print(f"{tag}:")
    print('  suggestion (raw):', x_star)
    print('  formatted:       ', suggestions[tag]['formatted'])
    print('  GP mean:          {:.6f}'.format(m_star))
    print('  GP std:           {:.6f}'.format(s_star))
    print()

suggestions


F1:
  suggestion (raw): [0.80323536 0.71943835]
  formatted:        0.803235-0.719438
  GP mean:          -0.000058
  GP std:           0.000992

F2:
  suggestion (raw): [0.95738716 0.92303323]
  formatted:        0.957387-0.923033
  GP mean:          0.382364
  GP std:           0.191318

F3:
  suggestion (raw): [0.00930784 0.06282507 0.00102808]
  formatted:        0.009308-0.062825-0.001028
  GP mean:          -0.076258
  GP std:           0.085480

F4:
  suggestion (raw): [0.39800271 0.39570842 0.36125371 0.44225652]
  formatted:        0.398003-0.395708-0.361254-0.442257
  GP mean:          -0.699524
  GP std:           0.192320

F5:
  suggestion (raw): [0.99741001 0.04175174 0.79179629 0.9749789 ]
  formatted:        0.997410-0.041752-0.791796-0.974979
  GP mean:          3592.366727
  GP std:           574.114683

F6:
  suggestion (raw): [0.49344941 0.12706393 0.63147756 0.90496357 0.00806487]
  formatted:        0.493449-0.127064-0.631478-0.904964-0.008065
  GP mean:          -

{'F1': {'x': array([0.80323536, 0.71943835]),
  'formatted': '0.803235-0.719438',
  'mean_pred': -5.772579271980495e-05,
  'std_pred': 0.000992197910167781},
 'F2': {'x': array([0.95738716, 0.92303323]),
  'formatted': '0.957387-0.923033',
  'mean_pred': 0.3823638533532368,
  'std_pred': 0.1913181445566698},
 'F3': {'x': array([0.00930784, 0.06282507, 0.00102808]),
  'formatted': '0.009308-0.062825-0.001028',
  'mean_pred': -0.07625817346313468,
  'std_pred': 0.0854799413977353},
 'F4': {'x': array([0.39800271, 0.39570842, 0.36125371, 0.44225652]),
  'formatted': '0.398003-0.395708-0.361254-0.442257',
  'mean_pred': -0.6995240292479679,
  'std_pred': 0.192320404063816},
 'F5': {'x': array([0.99741001, 0.04175174, 0.79179629, 0.9749789 ]),
  'formatted': '0.997410-0.041752-0.791796-0.974979',
  'mean_pred': 3592.3667266281236,
  'std_pred': 574.1146832639884},
 'F6': {'x': array([0.49344941, 0.12706393, 0.63147756, 0.90496357, 0.00806487]),
  'formatted': '0.493449-0.127064-0.631478-0.9